In [3]:
from abc import ABC, abstractmethod
from collections import Counter
from typing import Protocol

from text import texts

# import tiktoken


In [4]:
class TokenizerProtocol(Protocol):
    def encode(self, text: str) -> list[int]: ...
    def decode(self, tokens: list[int]) -> str: ...

In [5]:
class Tokenizer(ABC):
    def __init__(
        self,
        bos_str: str = "<BOS>",
        eos_str: str = "<EOS>",
        bos_id: int = 0,
        eos_id: int = 1,
    ):
        self._bos_str = bos_str
        self._eos_str = eos_str

        self._bos_id = bos_id
        self._eos_id = eos_id

        self._piece_to_token: dict[str, int] = {
            bos_str: bos_id,
            eos_str: eos_id,
        }

        self._token_to_piece: dict[int, str] = {
            bos_id: bos_str,
            eos_id: eos_str,
        }

    @abstractmethod
    def encode(self, text: str) -> list[int]:
        pass

    @abstractmethod
    def decode(self, tokens: list[int]) -> str:
        pass

    @property
    def vocab_size(self):
        return len(self._piece_to_token)

In [ ]:
class BPETokenizer(Tokenizer):
    def __init__(self, vocab_size: int = 100, **kwargs):
        super().__init__(**kwargs)

        self._target_vocab_size = vocab_size
        self._merges: dict[tuple[str, str], int] = {}

    def _get_pairs(self, symbols: list[str]) -> set[tuple[str, str]]:

        pairs = set()

        for i in range(len(symbols) - 1):
            pairs.add((symbols[i], symbols[i + 1]))

        return pairs

    def _get_most_frequent_pair(self, words: list[list[str]]):
        counter = Counter()
        for word in words:
            pairs = self._get_pairs(word)

            for pair in pairs:
                counter[pair] += 1

        if not counter:
            return None

        return counter.most_common(1)[0][0]

    def _merge_pair(self, symbols: list[str], pair: tuple[str, str]):
        result = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                result.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                result.append(symbols[i])
                i += 1

        return result

    def train(self, texts: list[str]):
        words = []

        for text in texts:
            print(1)
            for word in text.split():
                print
                symbols = list(word)

                words.append(symbols)

                for symbol in symbols:
                    if symbol not in self._piece_to_token:
                        token_id = len(self._piece_to_token)

                        self._piece_to_token[symbol] = token_id

                        self._token_to_piece[token_id] = symbol

        while self.vocab_size < self._target_vocab_size:
            pair = self._get_most_frequent_pair(words)
            if pair is None:
                break
            rank = len(self._merges)
            self._merges[pair] = rank
            new_token = "".join(pair)

            if new_token not in self._piece_to_token:
                token_id = len(self._piece_to_token)

                self._piece_to_token[new_token] = token_id

                self._token_to_piece[token_id] = new_token
            for i, word in enumerate(words):
                words[i] = self._merge_pair(word, pair)

    def _apply_bpe(self, symbols: list[str]):
        while True:
            pairs = self._get_pairs(symbols)
            if not pairs:
                break

            available = [pair for pair in pairs if pair in self._merges]
            if not available:
                break

            best_pair = min(available, key=lambda x: self._merges[x])
            symbols = self._merge_pair(symbols, best_pair)

        return symbols

    def encode(self, text: str) -> list[int]:
        tokens = [self._bos_id]
        for word in text.split():
            symbols = list(word)
            symbols = self._apply_bpe(symbols)
            for symbol in symbols:
                tokens.append(self._piece_to_token[symbol])
        tokens.append(self._eos_id)

        return tokens

    def decode(self, tokens: list[int]) -> str:
        pieces = []
        for token in tokens:
            if token in (self._bos_id, self._eos_id):
                continue
            pieces.append(self._token_to_piece[token])

        return " ".join(pieces)

In [15]:
tokenizer = BPETokenizer(vocab_size=30)
tokenizer.train(texts)
print(tokenizer._merges)
ids = tokenizer.encode("")
print(ids)
print(tokenizer.decode(ids))

1
1
1
1
1
1
1
1
{}
[0, 1]

